# Exam Score Prediction Demo

This notebook demonstrates the reproducible capstone workflow for predicting missing examination component scores. It uses the same backend services as the web application so that notebook evidence aligns with application outputs.

## Workflow Covered

1. Load dataset
2. Dataset inspection
3. Detection summary
4. Metadata recovery
5. Privacy-preserving preprocessing
6. Data cleaning
7. Exploratory Data Analysis
8. Feature engineering
9. Mode A benchmarking
10. Mode B controlled validation
11. Model comparison
12. SHAP / feature importance
13. Export generation
14. Summary of results

In [ ]:
from pathlib import Path
import sys
import json
import math

import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parents[1]
elif ROOT.name == 'backend':
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from backend.app.schemas.pipeline import MaxScoreMetadata, ProcessingRequest, PredictionMode
from backend.app.services.csv_parser import parse_examination_csv
from backend.app.services.pipeline import detect_upload, run_pipeline
from backend.app.services.anonymization import anonymize_dataset
from backend.app.ml.preprocessing.cleaning import clean_dataset
from backend.app.services.privacy_exports import build_public_research_dataset, private_subject_mapping_path
from backend.app.config.settings import get_settings

DATASET = ROOT / 'clean_sample_data.csv'
PREDICTION_DATASET = ROOT / 'Sample Prediction Data.csv'
REPORTS = ROOT / 'reports'
REPORTS.mkdir(exist_ok=True)
DATASET.exists(), PREDICTION_DATASET.exists()

## 1. Load Dataset and Inspect Structure

In [ ]:
raw_df, detected_maxima = parse_examination_csv(DATASET.read_bytes())
display(raw_df.head())
raw_df.shape, raw_df.columns.tolist(), detected_maxima

## 2. Detection Summary

In [ ]:
detection = detect_upload(DATASET.read_bytes(), DATASET.name)
pd.DataFrame(detection['subjects']).head(), detection['row_count'], detection['sensitive_fields']

## 3. Metadata Recovery

Paper maxima are required metadata. If maxima are not embedded in the uploaded file, they must be supplied explicitly. This cell loads previously extracted PDF metadata when available.

In [ ]:
metadata_path = REPORTS / 'pdf_extracted_subject_metadata.csv'
if metadata_path.exists():
    metadata = pd.read_csv(metadata_path, dtype={'subject_code': str})
else:
    metadata = pd.DataFrame(columns=['subject_code', 'subject_name', 'paper_count', 'p1_max', 'p2_max', 'p3_max', 'p4_max'])

subjects = sorted(raw_df['subject_code'].astype(str).str.strip().unique()) if 'subject_code' in raw_df else []
metadata = metadata[metadata['subject_code'].astype(str).isin(subjects)].copy()
max_scores = []
for _, row in metadata.iterrows():
    payload = {'subject_code': str(row['subject_code'])}
    for col in ['p1_max', 'p2_max', 'p3_max', 'p4_max']:
        if pd.notna(row.get(col)):
            payload[col] = float(row[col])
    max_scores.append(MaxScoreMetadata(**payload))
len(max_scores), metadata.head()

## 4. Privacy-Preserving Preprocessing and Cleaning

In [ ]:
anonymized = anonymize_dataset(raw_df)
cleaned = clean_dataset(anonymized, detected_max_scores=detected_maxima, max_scores=max_scores, require_complete_scores=True)
cleaning_summary = {
    'clean_rows': len(cleaned.data),
    'invalid_rows': len(cleaned.invalid_records),
    'absent_rows': len(cleaned.absent_records),
    'warnings': cleaned.warnings,
    'errors': cleaned.errors,
}
cleaning_summary

## 5. Public Dataset Export

The public dataset contains `subject_id` instead of original subject codes. The private mapping is written only to `Working Documents/`.

In [ ]:
settings = get_settings()
public_dataset, private_mapping = build_public_research_dataset(cleaned.data, private_subject_mapping_path(settings.project_root))
display(public_dataset.head())
public_dataset.columns.tolist(), private_subject_mapping_path(settings.project_root)

## 6. Exploratory Data Analysis and Feature Engineering

In [ ]:
score_columns = ['p1_score', 'p2_score', 'p3_score', 'p4_score']
display(cleaned.data[score_columns + ['partial_total', 'mean_score', 'score_spread', 'score_std', 'mean_normalized_score']].describe())

## 7. Mode A Benchmarking

In [ ]:
mode_a_response = run_pipeline(DATASET.read_bytes(), DATASET.name, ProcessingRequest(mode=PredictionMode.mode_a, max_scores=max_scores))
mode_a_response.summary, list(mode_a_response.exports.keys())

## 8. Model Comparison and Explainability

In [ ]:
metrics = pd.DataFrame(mode_a_response.metrics)
rankings = pd.DataFrame(mode_a_response.rankings)
display(metrics.sort_values(['RMSE', 'MAE']).head(10))
mode_a_response.plots.keys()

## 9. Mode B Controlled Validation

In [ ]:
prediction_df, prediction_maxima = parse_examination_csv(PREDICTION_DATASET.read_bytes())
subject_code = str(prediction_df['subject_code'].dropna().iloc[0])
paper_count = int(subject_code[-1])
rng = np.random.default_rng(42)
controlled = prediction_df.copy().astype(object)
available = list(prediction_df.index)
hidden = []
for target in [f'p{i}_score' for i in range(1, paper_count + 1)]:
    valid = [idx for idx in available if pd.notna(prediction_df.loc[idx, target])]
    selected = sorted(rng.choice(valid, size=max(1, round(len(prediction_df) * 0.10)), replace=False).tolist())
    for idx in selected:
        hidden.append({'row_index': idx, 'candidate_number': str(prediction_df.loc[idx, 'candidate_number']), 'target': target, 'actual': float(prediction_df.loc[idx, target])})
        controlled.loc[idx, target] = 'missing'
        available.remove(idx)
controlled_path = REPORTS / 'notebook_mode_b_controlled.csv'
controlled.to_csv(controlled_path, index=False)
max_payload = {'subject_code': subject_code, **{k: float(v) for k, v in prediction_maxima.items() if v is not None}}
mode_b_response = run_pipeline(controlled_path.read_bytes(), controlled_path.name, ProcessingRequest(mode=PredictionMode.mode_b, max_scores=[MaxScoreMetadata(**max_payload)]))
completed = pd.read_csv(mode_b_response.exports['completed_prediction_file'])
comparison = []
for item in hidden:
    row = completed[completed['candidate_number'].astype(str) == item['candidate_number']].iloc[0]
    predicted = float(row[item['target']])
    comparison.append({**item, 'predicted': predicted, 'error': predicted - item['actual'], 'abs_error': abs(predicted - item['actual'])})
comparison = pd.DataFrame(comparison)
comparison

## 10. Mode B Performance Summary

In [ ]:
mode_b_summary = {
    'hidden_rows': len(comparison),
    'MAE': mean_absolute_error(comparison['actual'], comparison['predicted']),
    'RMSE': math.sqrt(mean_squared_error(comparison['actual'], comparison['predicted'])),
    'R2': r2_score(comparison['actual'], comparison['predicted']),
    'prediction_status_distribution': completed['prediction_status'].value_counts().to_dict(),
}
mode_b_summary

## 11. Export Generation and Final Summary

The web application and notebook write generated files to `data/exports/` and `reports/`. Confidential source files and private mappings remain ignored by git.